# Random Forest – Regrese cen diamantů (Diamonds Price Prediction)
Tento Jupyter sešit představuje kompletní vypracované řešení cvičení **Random Forest - regression** ze 3. dne kurzu Machine Learning.

### Zadání cvičení:
1. Načíst data `diamonds_preprocessed.csv` pomocí knihovny Pandas.
2. Importovat potřebné třídy z knihovny Scikit-learn pro Random Forest regresi a evaluaci.
3. Rozdělit dataset na trénovací a testovací sadu v poměru 70:30 (`random_state=42`).
4. Vytvořit instanci modelu `RandomForestRegressor` s parametry `random_state=42` a `n_jobs=-1`.
5. Definovat slovník hyperparametrů `params` pro klíče `max_depth`, `min_samples_leaf` a `n_estimators` (2 až 4 hodnoty pro každý klíč).
6. Vytvořit objekt `RandomizedSearchCV` pro nalezení nejlepší kombinace s metrikou `mean_squared_error` (`neg_mean_squared_error`).
7. Natrénovat model s nejlepší sadou hyperparametrů na trénovacích datech `X_train, y_train`.
8. Provést predikci na testovací sadě `X_test`.
9. Vypočítat střední absolutní chybu (MAE – Mean Absolute Error) na testovací sadě a vyhodnotit model.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)
print('Knihovny úspěšně importovány.')

Knihovny úspěšně importovány.


In [2]:
# 1. Načtení dat diamonds_preprocessed.csv
data_path = '../data/MAL_downloadable materials_session 2/Day 3/diamonds_preprocessed.csv'
if not os.path.exists(data_path):
    data_path = 'diamonds_preprocessed.csv'

diamonds = pd.read_csv(data_path)
print('Rozměry datasetu diamantů:', diamonds.shape)
print('Sloupce:', diamonds.columns.tolist())
diamonds.head()

Rozměry datasetu diamantů: (53899, 10)
Sloupce: ['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y', 'z']


In [3]:
# 2. & 3. Příprava X, y a split 70:30 (random_state=42)
X = diamonds.drop('price', axis=1)
y = diamonds['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f'Trénovací sada: {X_train.shape}, Testovací sada: {X_test.shape}')

Trénovací sada: (37729, 9), Testovací sada: (16170, 9)


In [4]:
# 4., 5., 6. & 7. RandomForestRegressor + Mřížka params + RandomizedSearchCV (scoring='neg_mean_squared_error')
rf_regressor = RandomForestRegressor(random_state=42, n_jobs=-1)

params = {
    'max_depth': [10, 15, 20, None],
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50, 100, 150]
}

random_search = RandomizedSearchCV(
    estimator=rf_regressor,
    param_distributions=params,
    scoring='neg_mean_squared_error',
    n_iter=8,
    cv=3,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_
print('RandomizedSearchCV dokončen.')
print('Nejlepší nalezené hyperparametry:', random_search.best_params_)
best_rmse = (-random_search.best_score_)**0.5
print(f'Nejlepší CV RMSE: {best_rmse:.2f} USD')

RandomizedSearchCV dokončen.
Nejlepší nalezené hyperparametry: {'n_estimators': 100, 'min_samples_leaf': 2, 'max_depth': 15}
Nejlepší CV RMSE: 565.14 USD


In [5]:
# 8. & 9. Predikce na testovací sadě a výpočet MAE, RMSE a R²
y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5
r2 = r2_score(y_test, y_pred)

print('=== VÝSLEDKY NA TESTOVACÍ SADĚ (16 170 diamantů) ===')
print(f'Průměrná absolutní chyba (MAE): {mae:.2f} USD')
print(f'Střední kvadratická chyba (RMSE): {rmse:.2f} USD')
print(f'Koeficient determinace (R²): {r2:.4f} ({r2*100:.2f} % vysvětleného rozptylu)')

=== VÝSLEDKY NA TESTOVACÍ SADĚ (16 170 diamantů) ===
Průměrná absolutní chyba (MAE): 268.21 USD
Střední kvadratická chyba (RMSE): 535.59 USD
Koeficient determinace (R²): 0.9812 (98.12 % vysvětleného rozptylu)


In [6]:
# Vizualizace 1: Skutečná cena vs. Predikce modelu
plt.figure(figsize=(7, 7))
plt.scatter(y_test.iloc[:1500], y_pred[:1500], alpha=0.3, color='#10b981', s=15, label='Diamanty (vzorek 1500)')
plt.plot([0, 19000], [0, 19000], 'r--', lw=2, label='Ideální predikce (y = ŷ)')
plt.title('Skutečná cena vs. Predikce Random Forestu', fontsize=13)
plt.xlabel('Skutečná cena (USD)')
plt.ylabel('Predikovaná cena (USD)')
plt.legend()
plt.tight_layout()
plt.show()

In [7]:
# Vizualizace 2: Důležitost příznaků (Feature Importances)
feat_importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 5))
feat_importances.plot(kind='barh', color='#3b82f6')
plt.title('Důležitost příznaků (Feature Importance – MDI)', fontsize=13)
plt.xlabel('Relativní významnost')
plt.tight_layout()
plt.show()

### Závěrečné shrnutí a interpretace:
- **Výsledná průměrná absolutní chyba (MAE):** **268.21 USD**.
- **Srovnání s 1. dnem:**
  - Lineární regrese (OLS): MAE $\approx 740\text{ USD}$
  - Polynomiální regrese (stupeň 2): MAE $\approx 500\text{ USD}$
  - Samostatný rozhodovací strom (Decision Tree): MAE $= 358.34\text{ USD}$
  - **Random Forest (100 stromů):** MAE $= \mathbf{268.21\text{ USD}}$ (pokles chyby o **25.2 %** oproti samostatnému stromu a o **63.8 %** oproti OLS!).
- **Klíčové faktory ceny:** Hmotnost v karátech (`carat`) a prostorové rozměry (`y`, `x`) tvoří přes 85 % predikční síly, následované čistotou (`clarity`) a barvou (`color`).